# EDA — BTC-USD Volatility Spike Classifier

**Milestone 2** | Exploratory analysis of the feature table produced by `scripts/replay.py`.

Sections:
1. Load & inspect data
2. Distribution plots
3. Percentile analysis → choose τ
4. Time-series plot of rolling volatility
5. Class balance
6. Feature correlation heatmap
7. Evidently drift report (early vs. late)

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

PARQUET = "../data/processed/features.parquet"
REPORTS = Path("../reports/evidently")
REPORTS.mkdir(parents=True, exist_ok=True)

## 1. Load & Inspect Data

In [ ]:
df = pd.read_parquet(PARQUET)
df = df.sort_values("time").reset_index(drop=True)

# Convert Unix timestamp to datetime for plotting
df["datetime"] = pd.to_datetime(df["time"], unit="s", utc=True)

print(f"Shape : {df.shape}")
print(f"Period: {df['datetime'].min()}  →  {df['datetime'].max()}")
df.info()

In [ ]:
df.describe().T.style.background_gradient(cmap="Blues", axis=1)

In [ ]:
# Missing value check
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.any() else "None — clean dataset")

## 2. Distribution Plots

In [ ]:
feature_cols = [
    "rolling_vol_60s", "rolling_vol_30s", "spread_pct",
    "return_1s", "trade_intensity", "bid_ask_imbalance"
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flat, feature_cols):
    data = df[col].dropna()
    # Clip extreme outliers for readability
    lo, hi = data.quantile(0.01), data.quantile(0.99)
    ax.hist(data.clip(lo, hi), bins=80, edgecolor="none", alpha=0.8)
    ax.set_title(col, fontsize=11)
    ax.set_xlabel("value")
    ax.set_ylabel("count")
plt.suptitle("Feature Distributions (1st–99th percentile)", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("../reports/feature_distributions.png", bbox_inches="tight")
plt.show()

## 3. Percentile Analysis → Choose τ

Plot the 80th–99th percentiles of `sigma_future` to select the threshold τ.  
**Rule of thumb:** choose τ so that ~5–10% of windows are labeled `1`.

In [ ]:
if "sigma_future" not in df.columns:
    print("sigma_future not found — run scripts/replay.py first to generate labels")
else:
    pcts = np.arange(80, 100, 0.5)
    vals = np.percentile(df["sigma_future"].dropna(), pcts)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(pcts, vals, lw=2, color="steelblue")
    ax.axvline(92, color="tomato", linestyle="--", label="92nd pct (chosen τ)")
    tau_chosen = float(np.percentile(df["sigma_future"].dropna(), 92))
    ax.axhline(tau_chosen, color="tomato", linestyle=":")
    ax.set_xlabel("Percentile", fontsize=12)
    ax.set_ylabel("σ_future", fontsize=12)
    ax.set_title("σ_future Percentile Plot — Threshold Selection", fontsize=13)
    ax.legend()
    ax.text(92.3, tau_chosen * 1.02, f"τ = {tau_chosen:.5f}", color="tomato", fontsize=10)
    plt.tight_layout()
    plt.savefig("../reports/tau_percentile_plot.png", bbox_inches="tight")
    plt.show()
    print(f"\nChosen τ = {tau_chosen:.6f}  (92nd percentile)")
    print(f"Positive rate at τ: {(df['sigma_future'] >= tau_chosen).mean()*100:.1f}%")

## 4. Time-Series Plot — Rolling Volatility

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(df["datetime"], df["midprice"], lw=0.6, color="steelblue")
axes[0].set_ylabel("Midprice (USD)")
axes[0].set_title("BTC-USD Midprice")

axes[1].plot(df["datetime"], df["rolling_vol_60s"], lw=0.8, color="darkorange", label="60s vol")
axes[1].plot(df["datetime"], df["rolling_vol_30s"], lw=0.6, color="gold", alpha=0.7, label="30s vol")
if "sigma_future" in df.columns:
    tau_line = float(np.percentile(df["sigma_future"].dropna(), 92))
    axes[1].axhline(tau_line, color="tomato", linestyle="--", lw=1, label=f"τ={tau_line:.5f}")
axes[1].set_ylabel("Realized Vol")
axes[1].set_title("Rolling Realized Volatility")
axes[1].legend(fontsize=9)

axes[2].plot(df["datetime"], df["spread_pct"], lw=0.6, color="mediumseagreen")
axes[2].set_ylabel("Spread %")
axes[2].set_title("Bid-Ask Spread (%)")
axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

plt.suptitle("BTC-USD Time Series", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("../reports/time_series.png", bbox_inches="tight")
plt.show()

## 5. Class Balance

In [ ]:
if "label" not in df.columns:
    print("label column not found — run scripts/replay.py with --tau flag")
else:
    counts = df["label"].value_counts().sort_index()
    pct = df["label"].mean() * 100

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.bar(["No spike (0)", "Spike (1)"], counts.values,
           color=["steelblue", "tomato"], edgecolor="white")
    ax.set_ylabel("Count")
    ax.set_title(f"Class Balance  |  Positive rate = {pct:.1f}%")
    for i, v in enumerate(counts.values):
        ax.text(i, v + 50, f"{v:,}", ha="center", fontsize=10)
    plt.tight_layout()
    plt.savefig("../reports/class_balance.png", bbox_inches="tight")
    plt.show()

    print(f"Label=0 : {counts[0]:,}  ({100-pct:.1f}%)")
    print(f"Label=1 : {counts[1]:,}  ({pct:.1f}%)")
    print(f"\nImbalance ratio: 1 : {counts[0]/counts[1]:.1f}")

## 6. Feature Correlation Heatmap

In [ ]:
corr_cols = [
    "spread_pct", "return_1s", "rolling_vol_30s", "rolling_vol_60s",
    "trade_intensity", "bid_ask_imbalance"
]
if "label" in df.columns:
    corr_cols.append("label")

corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title("Feature Correlation Matrix", fontsize=13)
plt.tight_layout()
plt.savefig("../reports/correlation_heatmap.png", bbox_inches="tight")
plt.show()

## 7. Evidently Drift Report — Early vs. Late Data

In [ ]:
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, DataQualityPreset

drift_cols = [
    "spread_pct", "return_1s", "rolling_vol_30s", "rolling_vol_60s",
    "trade_intensity", "bid_ask_imbalance"
]

n = len(df)
df_early = df.iloc[:n // 2][drift_cols].reset_index(drop=True)
df_late  = df.iloc[n // 2:][drift_cols].reset_index(drop=True)

report = Report(metrics=[DataDriftPreset(), DataQualityPreset()])
report.run(reference_data=df_early, current_data=df_late)

report_path = str(REPORTS / "drift_report.html")
report.save_html(report_path)
print(f"Drift report saved → {report_path}")